In [1]:
import os

In [2]:
import gradio as gr
from openai import OpenAI
import sqlite3
import json

c:\Users\Mrunal\AIProjects\Project2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODEL = 'qwen2.5:3b'
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [4]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [5]:
DB = "prices.db"
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [6]:
get_ticket_price("Paris")

DATABASE TOOL CALLED: Getting price for Paris


'Ticket price to Paris is $500.0'

In [7]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [8]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [9]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [10]:
import base64
from io import BytesIO
from PIL import Image

In [ ]:
# def artist(city):
#     image_response = ollama.images.generate(
#             model="qwen2.5:3b",
#             prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
#             size="1024x1024",
#             n=1,
#             response_format="b64_json",
#         )
#     image_base64 = image_response.data[0].b64_json
#     image_data = base64.b64decode(image_base64)
#     return Image.open(BytesIO(image_data))

In [13]:
from diffusers import StableDiffusionPipeline
import torch

def artist(city):
    pipe = StableDiffusionPipeline.from_pretrained(
        "OFA-Sys/small-stable-diffusion-v0",
        # torch_dtype=torch.float16
    )
    pipe = pipe.to("cpu")  # use "cpu" if no GPU
    
    image = pipe(
        f"A vacation in {city}, tourist spots, vibrant pop-art style"
    ).images[0]
    
    return image

image = artist("New York City")
image.save("output.png")
image.show()

Loading pipeline components...:  29%|██▊       | 2/7 [00:00<00:02,  2.09it/s]An error occurred while trying to fetch C:\Users\Mrunal\.cache\huggingface\hub\models--OFA-Sys--small-stable-diffusion-v0\snapshots\38e10e5e71e8fbf717a47a81e7543cd01c1a8140\vae: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\Mrunal\.cache\huggingface\hub\models--OFA-Sys--small-stable-diffusion-v0\snapshots\38e10e5e71e8fbf717a47a81e7543cd01c1a8140\vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:  43%|████▎     | 3/7 [00:00<00:01,  3.18it/s]An error occurred while trying to fetch C:\Users\Mrunal\.cache\huggingface\hub\models--OFA-Sys--small-stable-diffusion-v0\snapshots\38e10e5e71e8fbf717a47a81e7543cd01c1a8140\unet: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\Mrunal\.cache\huggingface\hub\models--OFA-Sys--small-stable-diffusion-v0\snapshots\38e10e5e71e8fbf717a4

In [14]:
# image = artist("New York City")
# display(image)

In [ ]:
# import pyttsx3

# def talker(message):
#     engine = pyttsx3.init()
#     engine.say(message)
#     engine.runAndWait()

In [27]:
from gtts import gTTS
# import os

def talker(message):
    tts = gTTS(text=message, lang='en')
    tts.save("output.mp3")
    os.system("start output.mp3")  # windows command to play audio
    with open("output.mp3", "rb") as f:
        return f.read()

In [28]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])
    
    return history, voice, image

In [29]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

In [ ]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500)
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\Users\Mrunal\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "C:\Users\Mrunal\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host


DATABASE TOOL CALLED: Getting price for Tokyo


Loading pipeline components...:  29%|██▊       | 2/7 [00:00<00:02,  1.98it/s]An error occurred while trying to fetch C:\Users\Mrunal\.cache\huggingface\hub\models--OFA-Sys--small-stable-diffusion-v0\snapshots\38e10e5e71e8fbf717a47a81e7543cd01c1a8140\vae: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\Mrunal\.cache\huggingface\hub\models--OFA-Sys--small-stable-diffusion-v0\snapshots\38e10e5e71e8fbf717a47a81e7543cd01c1a8140\vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:  43%|████▎     | 3/7 [00:01<00:01,  2.66it/s]An error occurred while trying to fetch C:\Users\Mrunal\.cache\huggingface\hub\models--OFA-Sys--small-stable-diffusion-v0\snapshots\38e10e5e71e8fbf717a47a81e7543cd01c1a8140\unet: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\Mrunal\.cache\huggingface\hub\models--OFA-Sys--small-stable-diffusion-v0\snapshots\38e10e5e71e8fbf717a4